# Task 5 — Step 3: Web Snippet Extraction + Labeling

For each sampled entity (Step 1) and each of its top-`TOP_K_RESULTS` Tavily search results
(Step 2), calls Claude `claude-sonnet-4-6` to:

1. Decide whether the web result describes a specific record that can be extracted into the
   dataset's schema (`relevant`).
2. If relevant, extract a structured record with the dataset's columns
   (`WDC_COLS` / `DBLP_COLS`) from the web snippet.
3. Label whether the extracted record refers to the same real-world entity as the original
   sampled entity (`label` 0/1).

**Output**: `data/processed/<dataset>/web_labeled.jsonl` — one JSON record per
(entity, web result) candidate, including `relevant=false` records (kept for resumability).
Step 4 filters to `relevant=true` records to build `train_aug_web.txt`.

**Resumable**: re-running skips `(id, side, url)` candidates already processed.

Change only `DATASET` to switch between datasets.

## Configuration

In [9]:
DATASET = "wdc-products"      # "dblp-scholar" | "wdc-products"
TOP_K_RESULTS = 3             # use the top-N Tavily results per entity
MAX_CONTENT_LEN = 800          # truncate web snippet content to this many characters
LLM_MODEL = "claude-sonnet-4-6"

## Imports & paths

In [10]:
import json
import sys
import time
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

from src.data_prep.preprocess import serialize_record, WDC_COLS, DBLP_COLS

PROCESSED = ROOT / "data" / "processed" / DATASET
TAVILY_RESULTS = PROCESSED / "tavily_results.jsonl"
OUTPUT = PROCESSED / "web_labeled.jsonl"

load_dotenv(ROOT / ".env")
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env

COLS = WDC_COLS if DATASET == "wdc-products" else DBLP_COLS

print(f"Dataset    : {DATASET}")
print(f"Columns    : {COLS}")
print(f"Tavily in  : {TAVILY_RESULTS}")
print(f"Output     : {OUTPUT}")

Dataset    : wdc-products
Columns    : ['brand', 'title', 'description', 'price', 'priceCurrency']
Tavily in  : /Users/abd/Developer/thesis-project/data/processed/wdc-products/tavily_results.jsonl
Output     : /Users/abd/Developer/thesis-project/data/processed/wdc-products/web_labeled.jsonl


## Load Tavily results and build candidate list

In [11]:
tavily_records = [json.loads(l) for l in open(TAVILY_RESULTS)]
print(f"Entities with web results: {len(tavily_records)}")

candidates = []
for rec in tavily_records:
    for r in rec["results"][:TOP_K_RESULTS]:
        candidates.append((rec, r))

print(f"Total (entity, web result) candidates (top-{TOP_K_RESULTS}): {len(candidates)}")

Entities with web results: 300
Total (entity, web result) candidates (top-3): 900


## Prompt builder and extraction+labeling function

In [12]:
def build_prompt(original_text: str, web_title: str, web_url: str, web_content: str) -> str:
    schema_desc = ", ".join(COLS)

    if DATASET == "wdc-products":
        entity_kind = "e-commerce product listing"
        match_desc = ("the exact same product (same model, variant, and specs — "
                       "not just the same product line)")
        attention = (
            "Pay close attention to: color, storage/memory capacity, size/dimensions, "
            "bundle contents, edition/version, and model number.\n"
        )
    else:
        entity_kind = "academic paper record"
        match_desc = "the exact same paper (same title, same authors, same publication)"
        attention = ""

    example_fields = ", ".join(f'"{c}": "..."' for c in COLS)

    return (
        f"You are extracting structured records from web search results for entity matching.\n\n"
        f"ORIGINAL RECORD ({entity_kind}):\n{original_text}\n\n"
        f"WEB SEARCH RESULT:\n"
        f"Title: {web_title}\n"
        f"URL: {web_url}\n"
        f"Content: {web_content}\n\n"
        f"Task:\n"
        f"1. Decide if the web search result describes a SPECIFIC {entity_kind} whose fields "
        f"({schema_desc}) can be extracted from the title/content above. "
        f"If it is irrelevant (e.g. a category/listing page, blog post, search results page, "
        f"or does not describe a specific record), respond with {{\"relevant\": false}} and "
        f"nothing else.\n"
        f"2. If relevant, extract a record with exactly these fields: {schema_desc} "
        f"(use \"\" for any field you cannot determine from the text).\n"
        f"3. Decide whether the extracted record refers to {match_desc} as the ORIGINAL RECORD "
        f"(label 1) or not (label 0).\n"
        f"{attention}"
        f"\nReply with valid JSON only — no markdown, no extra text. Examples:\n"
        f'{{"relevant": false}}\n'
        f'{{"relevant": true, "label": 1, "extracted": {{{example_fields}}}, "reasoning": "one sentence"}}'
    )


def extract_and_label(entity_id: int, side: str, bucket: str, original_text: str, web_result: dict) -> dict | None:
    web_title = web_result.get("title", "")
    web_url = web_result.get("url", "")
    web_content = web_result.get("content", "")[:MAX_CONTENT_LEN]
    prompt = build_prompt(original_text, web_title, web_url, web_content)

    for attempt in range(3):
        try:
            resp = client.messages.create(
                model=LLM_MODEL,
                max_tokens=400,
                messages=[{"role": "user", "content": prompt}],
            )
            result = json.loads(resp.content[0].text.strip())

            if not result.get("relevant", False):
                return {
                    "id": entity_id, "side": side, "bucket": bucket,
                    "url": web_url, "relevant": False,
                }

            web_text = serialize_record(result["extracted"], COLS)
            if side == "left":
                left_text, right_text = original_text, web_text
            else:
                left_text, right_text = web_text, original_text

            return {
                "id": entity_id, "side": side, "bucket": bucket,
                "url": web_url, "relevant": True,
                "left_text": left_text, "right_text": right_text,
                "label": int(result["label"]),
                "reasoning": str(result.get("reasoning", "")),
                "model": LLM_MODEL,
            }
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            if attempt == 2:
                print(f"  [warn] parse failed id={entity_id} side={side} url={web_url[:60]!r}: {type(e).__name__} — skipping")
                return None
            time.sleep(1)


print("Prompt and extraction/labeling functions defined.")
print("\nSample prompt (first 600 chars):")
sample_rec, sample_result = candidates[0]
print(build_prompt(sample_rec["text"], sample_result.get("title", ""),
                    sample_result.get("url", ""), sample_result.get("content", "")[:MAX_CONTENT_LEN])[:600])

Prompt and extraction/labeling functions defined.

Sample prompt (first 600 chars):
You are extracting structured records from web search results for entity matching.

ORIGINAL RECORD (e-commerce product listing):
COL brand VAL Brother COL title VAL Brother HL-L6300DW Business Laser Printer for Mid-Size Workgroups -HL-L6300DW COL description VAL The Brother HL-L6300DW is the ultimate monochrome laser printer for mid-sized workgroups with higher print volumes. This business-durable printer offers great value due to the included high-yield toner cartridge and offers even lower cost output due to the super high-yield replacement toner cartridge. Plus, maximize your productivity 


## Resume check — skip already-processed (id, side, url) candidates

In [13]:
labeled = []
already_done = set()   # (id, side, url)

if OUTPUT.exists():
    with open(OUTPUT) as f:
        for line in f:
            rec = json.loads(line)
            labeled.append(rec)
            already_done.add((rec["id"], rec["side"], rec["url"]))
    print(f"Resuming: {len(labeled)} candidates already processed")
else:
    print("Starting fresh — no existing output file found")

remaining = [(rec, r) for rec, r in candidates if (rec["id"], rec["side"], r.get("url", "")) not in already_done]
print(f"Remaining: {len(remaining)} candidates")

Starting fresh — no existing output file found
Remaining: 900 candidates


## Extraction + labeling loop

In [14]:
n_relevant = 0
n_irrelevant = 0
n_failed = 0

with open(OUTPUT, "a") as out_f:
    for i, (rec, r) in enumerate(remaining):
        result = extract_and_label(rec["id"], rec["side"], rec["bucket"], rec["text"], r)

        if result is None:
            n_failed += 1
            continue

        labeled.append(result)
        out_f.write(json.dumps(result) + "\n")
        out_f.flush()

        if result["relevant"]:
            n_relevant += 1
        else:
            n_irrelevant += 1

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(remaining)} processed "
                  f"(relevant={n_relevant}, irrelevant={n_irrelevant}, failed={n_failed})")

print(f"\nDone. Processed {len(remaining)} candidates: "
      f"{n_relevant} relevant, {n_irrelevant} irrelevant, {n_failed} failed.")

  50/900 processed (relevant=43, irrelevant=7, failed=0)
  100/900 processed (relevant=90, irrelevant=10, failed=0)
  150/900 processed (relevant=136, irrelevant=14, failed=0)
  200/900 processed (relevant=184, irrelevant=16, failed=0)
  250/900 processed (relevant=233, irrelevant=17, failed=0)
  300/900 processed (relevant=279, irrelevant=21, failed=0)
  350/900 processed (relevant=322, irrelevant=28, failed=0)
  400/900 processed (relevant=365, irrelevant=35, failed=0)
  450/900 processed (relevant=409, irrelevant=41, failed=0)
  500/900 processed (relevant=453, irrelevant=47, failed=0)
  550/900 processed (relevant=493, irrelevant=57, failed=0)
  600/900 processed (relevant=537, irrelevant=63, failed=0)
  650/900 processed (relevant=582, irrelevant=68, failed=0)
  700/900 processed (relevant=626, irrelevant=74, failed=0)
  750/900 processed (relevant=672, irrelevant=78, failed=0)
  800/900 processed (relevant=714, irrelevant=86, failed=0)
  850/900 processed (relevant=761, irrelevan

## Summary

In [15]:
all_records = [json.loads(l) for l in open(OUTPUT)]
relevant_records = [r for r in all_records if r["relevant"]]

n_pos = sum(1 for r in relevant_records if r["label"] == 1)
n_neg = len(relevant_records) - n_pos
pos_rate = n_pos / len(relevant_records) * 100 if relevant_records else 0

print(f"Total candidates processed : {len(all_records)}")
print(f"  Relevant (usable pairs)  : {len(relevant_records)} "
      f"({len(relevant_records)/len(all_records)*100:.1f}%)")
print(f"  Irrelevant (skipped)     : {len(all_records) - len(relevant_records)}")
print()
print(f"Usable pairs label distribution:")
print(f"  Matches (label=1)     : {n_pos}  ({pos_rate:.1f}%)")
print(f"  Non-matches (label=0) : {n_neg}  ({100-pos_rate:.1f}%)")

# Rough cost estimate: claude-sonnet-4-6 input ~$3/MTok, output ~$15/MTok
estimated_cost = len(all_records) * (800 * 3e-6 + 150 * 15e-6)
print(f"\nEstimated API cost: ~${estimated_cost:.3f}")

Total candidates processed : 900
  Relevant (usable pairs)  : 803 (89.2%)
  Irrelevant (skipped)     : 97

Usable pairs label distribution:
  Matches (label=1)     : 649  (80.8%)
  Non-matches (label=0) : 154  (19.2%)

Estimated API cost: ~$4.185


## Spot-check: sample extracted pairs

In [16]:
import re


def trunc(s: str, n: int = 100) -> str:
    s = re.sub(r"COL \w+ VAL ", " | ", str(s)).strip(" |")
    return s[:n] + "…" if len(s) > n else s


for label_val, label_name in [(1, "MATCHES"), (0, "NON-MATCHES")]:
    subset = [r for r in relevant_records if r["label"] == label_val][:5]
    print(f"\n{'='*70}")
    print(f"  {label_name} (showing up to 5)")
    print(f"{'='*70}")
    for r in subset:
        print(f"  id={r['id']} side={r['side']} bucket={r['bucket']!r}")
        print(f"  LEFT : {trunc(r['left_text'])}")
        print(f"  RIGHT: {trunc(r['right_text'])}")
        print(f"  url: {r['url']}")
        print(f"  reason: {r['reasoning']}")
        print()


  MATCHES (showing up to 5)
  id=49605449 side=left bucket='Brother'
  LEFT : Brother  | Brother HL-L6300DW Business Laser Printer for Mid-Size Workgroups -HL-L6300DW  | The Brot…
  RIGHT: Brother  | BROTHER HLL6300DW Business Laser Printer for Mid-Size Workgroups with Higher Print Volume…
  url: https://wisconsin-copiers.com/wisconsin/menomonee-falls/laser-printer-sales.php
  reason: The web result describes the same Brother HL-L6300DW model laser printer for mid-size workgroups, matching the model number and key features.

  id=49605449 side=left bucket='Brother'
  LEFT : Brother  | Brother HL-L6300DW Business Laser Printer for Mid-Size Workgroups -HL-L6300DW  | The Brot…
  RIGHT: Brother  | Brother HLL6300DW Business Laser Printer for Mid-Size Workgroups with Higher Print Volume…
  url: https://promtlocaltech.com/laser-printer-sales
  reason: The web search result lists the Brother HL-L6300DW, the same model number as the original record, described as a Business Laser Printer for M